## Importing Libraries

In [1]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.1.2
    Uninstalling pip-23.1.2:
      Successfully uninstalled pip-23.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tensorflow-metadata to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import files
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import image_classifier

import matplotlib.pyplot as plt

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


## Loading Dataset

In [3]:
#save final weights to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
zip_file_name = "/content/drive/MyDrive/ingredient_dataset/ingredient_list" #@param {type:"string"}
!unzip {zip_file_name}.zip

Archive:  /content/drive/MyDrive/ingredient_dataset/ingredient_list.zip
   creating: apple/
  inflating: __MACOSX/._apple        
  inflating: apple/Image_28.jpg      
  inflating: __MACOSX/apple/._Image_28.jpg  
  inflating: apple/Image_16.jpg      
  inflating: __MACOSX/apple/._Image_16.jpg  
  inflating: apple/Image_17.jpg      
  inflating: __MACOSX/apple/._Image_17.jpg  
  inflating: apple/Image_38.jpg      
  inflating: __MACOSX/apple/._Image_38.jpg  
  inflating: apple/Image_10.jpg      
  inflating: __MACOSX/apple/._Image_10.jpg  
  inflating: apple/Image_39.jpg      
  inflating: __MACOSX/apple/._Image_39.jpg  
  inflating: apple/Image_76.png      
  inflating: __MACOSX/apple/._Image_76.png  
  inflating: apple/Image_62.jpg      
  inflating: __MACOSX/apple/._Image_62.jpg  
  inflating: apple/Image_63.jpg      
  inflating: __MACOSX/apple/._Image_63.jpg  
  inflating: apple/Image_61.jpg      
  inflating: __MACOSX/apple/._Image_61.jpg  
  inflating: apple/Image_49.jpg      
  

In [14]:
image_path = '/content/ingredient_db'

In [15]:
print(image_path)
labels = []
for i in os.listdir(image_path):
  if os.path.isdir(os.path.join(image_path, i)):
    labels.append(i)
print(labels)

/content/ingredient_db
['corn', 'banana', 'carrot', 'tomato', 'lemon', 'garlic', 'cucumber', 'apple', 'potato', 'onion']


In [16]:
NUM_EXAMPLES = 5

for label in labels:
  label_dir = os.path.join(image_path, label)
  example_filenames = os.listdir(label_dir)[:NUM_EXAMPLES]
  fig, axs = plt.subplots(1, NUM_EXAMPLES, figsize=(10,2))
  for i in range(NUM_EXAMPLES):
    axs[i].imshow(plt.imread(os.path.join(label_dir, example_filenames[i])))
    axs[i].get_xaxis().set_visible(False)
    axs[i].get_yaxis().set_visible(False)
  fig.suptitle(f'Showing {NUM_EXAMPLES} examples for {label}')

plt.show()

In [17]:
data = image_classifier.Dataset.from_folder(image_path)
train_data, remaining_data = data.split(0.8)
test_data, validation_data = remaining_data.split(0.5)

## Model Training

In [18]:
spec = image_classifier.SupportedModels.EFFICIENTNET_LITE4
hparams = image_classifier.HParams(export_dir="exported_model",epochs=50)
options = image_classifier.ImageClassifierOptions(supported_model=spec, hparams=hparams)

In [19]:
model = image_classifier.ImageClassifier.create(
    train_data = train_data,
    validation_data = validation_data,
    options=options,
)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 1280)              11837936  
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 11850746 (45.21 MB)
Trainable params: 12810 (50.04 KB)
Non-trainable params: 11837936 (45.16 MB)
_________________________________________________________________
None


Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.BICUBIC...)` instead.


Epoch 1/50
337/337 [==============================] - 23s 50ms/step - loss: 2.4238 - accuracy: 0.0920 - val_loss: 2.2908 - val_accuracy: 0.1294
Epoch 2/50
337/337 [==============================] - 15s 45ms/step - loss: 2.1509 - accuracy: 0.2626 - val_loss: 1.8947 - val_accuracy: 0.4824
Epoch 3/50
337/337 [==============================] - 15s 44ms/step - loss: 1.6961 - accuracy: 0.6469 - val_loss: 1.4944 - val_accuracy: 0.8353
Epoch 4/50
337/337 [==============================] - 15s 44ms/step - loss: 1.3631 - accuracy: 0.7834 - val_loss: 1.2262 - val_accuracy: 0.8941
Epoch 5/50
337/337 [==============================] - 15s 44ms/step - loss: 1.1483 - accuracy: 0.8264 - val_loss: 1.0516 - val_accuracy: 0.9294
Epoch 6/50
337/337 [==============================] - 15s 44ms/step - loss: 1.0270 - accuracy: 0.8680 - val_loss: 0.9447 - val_accuracy: 0.9412
Epoch 7/50
337/337 [==============================] - 15s 44ms/step - loss: 0.9462 - accuracy: 0.8754 - val_loss: 0.8787 - val_accuracy:

##Model Evaluation

In [20]:
loss, acc = model.evaluate(test_data)
print(f'Test loss:{loss}, Test accuracy:{acc}')

3/3 [==============================] - 10s 1s/step - loss: 0.6809 - accuracy: 0.9286
Test loss:0.6808831691741943, Test accuracy:0.9285714030265808


##Model Conversion

In [21]:
model.export_model()

In [23]:
!ls exported_model
files.download('exported_model/model.tflite')

checkpoint  metadata.json  model.tflite  summaries


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Retraining parameters

> Indented block



You can further customize how the retraining process runs to adjust training time and potentially increase the retrained model's performance. *These parameters are optional*. Use the `ImageClassifierModelOptions` class and the `HParams` class to set these additional options.

Use the `ImageClassifierModelOptions` class parameters to customize the existing model. It has the following customizable parameter that affects model accuracy:
* `dropout_rate`: The fraction of the input units to drop. Used in dropout layer. Defaults to 0.05.

Use the `HParams` class to customize other parameters related to training and saving the model:

* `learning_rate`: The learning rate to use for gradient descent training. Defaults to 0.001.
* `batch_size`: Batch size for training. Defaults to 2.
* `epochs`: Number of training iterations over the dataset. Defaults to 10.
* `steps_per_epoch`: An optional integer that indicates the number of training steps per epoch. If not set, the training pipeline calculates the default steps per epoch as the training dataset size divided by batch size.
* `shuffle`: True if the dataset is shuffled before training. Defaults to False.
* `do_fine_tuning`: If true, the base module is trained together with the classification layer on top. This defaults to False, which means only the classification layer is trained and pre-trained weights for the base module are frozen.
* `l1_regularizer`: A regularizer that applies a L1 regularization penalty. Defaults to 0.0.
* `l2_regularizer`: A regularizer that applies a L2 regularization penalty. Defaults to 0.0001.
* `label_smoothing`: Amount of label smoothing to apply. See [`tf.keras.losses`](https://www.tensorflow.org/api_docs/python/tf/keras/losses) for more details. Defaults to 0.1.
* `do_data_augmentation`: Whether or not the training dataset is augmented by applying random transformations such as cropping, flipping, etc. See [utils.image_preprocessing](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image) for details. Defaults to True.
* `decay_samples`: Number of training samples used to calculate the decay steps and create the training optimizer. Defaults to 2,560,000.
* `warmup_epochs`: Number of warmup steps for a linear increasing warmup schedule on the learning rate. Used to set up warmup schedule by `model_util.WarmUp`. Defaults to 2.



In [ ]:
hparams=image_classifier.HParams(epochs=15, export_dir="exported_model_2")
options = image_classifier.ImageClassifierOptions(supported_model=spec, hparams=hparams)
model_2 = image_classifier.ImageClassifier.create(
    train_data = train_data,
    validation_data = validation_data,
    options=options,
)

In [ ]:
loss, accuracy = model_2.evaluate(test_data)

In [ ]:
model.export_model()